# 29 AG RUN-1: SMR value and required support to 2055 (descriptive)

**What this notebook does.** RUN-1 (`run1_smr100_{t}_p50_grow`, lab batch `ag`) keeps each P50 mandate growing to 2055 at
its 2041–2050 pace. Its comparison runs are paper 1's `hz_{t}_p50_ext` (batch `ratedesign`): the same samples run to 2055
with the SMR fleet flat after 2050. Through 2050 the two are the same run (O38). This notebook reports, for both runs and
every solve year 2041–2055, the quantities in Ethan's item-21 note (review page, `run1-readouts` = descriptive,
2026-11-16): the SMR value, the required support, and their change after 2050, with a few system indicators beside
them. It makes no claim and tests nothing (the paper is descriptive).

**Definitions (paper 1's, unchanged).**
- *Required support per kW* = the mandate's shadow price (`nuclear_cap_price`, 2004 $/MW) × `pvf_onm` (14.7645) ÷ 1000:
  the capital support a new SMR needs in that solve year. Zero when no mandate binds.
- *Value of new SMRs per kW-yr* = (`val_load` + `val_resmarg` + `val_rps`) of the year's new SMR MW, summed over regions
  (`valnew`), ÷ new SMR kW (paper 1's m10f quantity). Reported only for solve years with new SMR MW > 0; `_ext` builds
  none after 2050.
- *Fleet market revenue per kW-yr* = the SMR fleet's market revenue (`revenue`, categories load + res_marg + rps) ÷ SMR
  fleet kW (paper 1's V1 R_t). It exists in both runs after 2050.
- System indicators: load-weighted energy price (`P_E` ÷ `Q_E`), process CO2 (`emit_nat`), system cost (`systemcost`).
- Money in 2025 $ (`plotstyle.to2025`, the ledger's factor `TO2025_FROM_2004`); 2050 → 2055 changes in % of 2050.

In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():
    _p = _p.parent
sys.path.insert(0, str(_p))
import numpy as np
import pandas as pd
import nlpaper
from nlpaper import paths
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
OUT = paths.exports("29")
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
RUNS = {t: {"RUN-1": ("ag", f"run1_smr100_{t}_p50_grow"), "_ext": ("ratedesign", f"hz_{t}_p50_ext")} for t in TARGETS}
K = L.TO2025_FROM_2004
YEARS = [2041, 2044, 2047, 2050, 2053, 2055]

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


## 1. Readouts by run and solve year

In [2]:
def readouts(batch, case):
    years = np.array(sorted(L._sel(batch, "pvf_onm", case)["t"]))
    years = years[years >= 2026]
    by = lambda d: L._by_t(d, years)
    dual, pvf = by(L._sel(batch, "nuclear_cap_price", case)), by(L._sel(batch, "pvf_onm", case))
    v = L._sel(batch, "valnew", case)
    v = v[v["*.1"] == "nuclear-smr"]
    vm = lambda m: by(v[v["*"] == m])
    new_mw = by(L._sel(batch, "cap_new_ivrt", case).query("i == 'nuclear-smr'"))
    vn_mw = vm("MW")
    val = vm("val_load") + vm("val_resmarg") + vm("val_rps")
    cap = by(L._sel(batch, "cap_nat", case).query("i == 'nuclear-smr'"))
    rev = L._sel(batch, "revenue_nat", case)
    rev = rev[(rev["i"] == "nuclear-smr") & rev["rev_cat"].isin(["load", "res_marg", "rps"])]
    der = L._sel(batch, "derived_ledger", case).pivot_table(index="t", columns="item", values="Value", aggfunc="sum")
    der = der.reindex(years).fillna(0.0)
    em = L._sel(batch, "emit_nat", case).query("eall == 'CO2' and etype == 'process'")
    sc = L._sel(batch, "systemcost", case)
    out = pd.DataFrame({
        "SMR fleet, GW": cap / 1e3,
        "new SMR, GW": new_mw / 1e3,
        "required support, 2025 $/kW": dual * pvf / 1e3 * K,
        "value of new SMRs, 2025 $/kW-yr": (val / (vn_mw * 1e3)).where(vn_mw > 0) * K,
        "fleet revenue, 2025 $/kW-yr": (by(rev) / (cap * 1e3)).where(cap > 0) * K,
        "  of which energy": (by(rev[rev["rev_cat"] == "load"]) / (cap * 1e3)).where(cap > 0) * K,
        "  of which capacity": (by(rev[rev["rev_cat"] == "res_marg"]) / (cap * 1e3)).where(cap > 0) * K,
        "  of which RPS/CES": (by(rev[rev["rev_cat"] == "rps"]) / (cap * 1e3)).where(cap > 0) * K,
        "energy price, 2025 $/MWh": (der["P_E"] / der["Q_E"]).where(der["Q_E"] > 0) * K,
        "process CO2, Mt": by(em) / 1e6,
        "system cost, 2025 $B": by(sc) / 1e9 * K})
    # paper 1's new-unit MW in valnew equals the new SMR MW of cap_new_ivrt
    assert ((vn_mw - new_mw).abs() <= 1e-3 * new_mw.abs().clip(lower=1.0)).all(), (case, (vn_mw - new_mw).abs().max())
    return out


R = pd.concat({(t, w): readouts(*bc) for t, d in RUNS.items() for w, bc in d.items()}, names=["target", "run", "t"])
R.to_csv(OUT / "run1_readouts.csv")
for t in TARGETS:
    print(f"\n=== {t} P50")
    x = R.loc[t].reset_index()
    x = x[x["t"].isin(YEARS)].set_index(["t", "run"]).unstack("run").round(1)
    print(x.T.to_string())


=== eia P50
t                                             2041         2044    2047         2050         2053         2055
                                run                                                                           
SMR fleet, GW                   RUN-1    16.600000    16.800000    17.1    20.700000    22.100000    23.000000
                                _ext     16.600000    16.800000    17.1    20.700000    20.700000    20.700000
new SMR, GW                     RUN-1     6.700000     0.200000     0.2     3.600000     1.400000     0.900000
                                _ext      6.700000     0.200000     0.2     3.600000     0.000000     0.000000
required support, 2025 $/kW     RUN-1  5104.400000  4936.700000  4437.1  4698.600000  4612.500000  4520.200000
                                _ext   5104.400000  4936.600000  4437.3  4698.500000     0.000000     0.000000
value of new SMRs, 2025 $/kW-yr RUN-1   553.200000   432.300000   562.1   528.900000   563.700000  

## 2. Change after 2050

For each target: the 2050 level (equal in both runs within solver noise), each run's 2053 and 2055 levels, the change
2050 → 2055 in %, and RUN-1 − `_ext` in 2055. A blank means the quantity does not exist in that run and year (no new
SMR, so no new-unit value).

In [3]:
Q = ["SMR fleet, GW", "required support, 2025 $/kW", "value of new SMRs, 2025 $/kW-yr", "fleet revenue, 2025 $/kW-yr",
     "energy price, 2025 $/MWh", "process CO2, Mt", "system cost, 2025 $B"]
rows = []
for t in TARGETS:
    for q in Q:
        a, e = R.loc[(t, "RUN-1"), q], R.loc[(t, "_ext"), q]
        rows.append({"target": t, "quantity": q, "2050": a.get(2050), "RUN-1 2053": a.get(2053), "RUN-1 2055": a.get(2055),
                     "_ext 2053": e.get(2053), "_ext 2055": e.get(2055),
                     "RUN-1 2050->2055 %": 100 * (a.get(2055) / a.get(2050) - 1) if a.get(2050) else np.nan,
                     "_ext 2050->2055 %": 100 * (e.get(2055) / e.get(2050) - 1) if e.get(2050) else np.nan,
                     "RUN-1 - _ext 2055": a.get(2055) - e.get(2055),
                     "2050 RUN-1 - _ext": a.get(2050) - e.get(2050)})
CH = pd.DataFrame(rows)
CH.to_csv(OUT / "run1_change.csv", index=False)
for q in Q:
    print(f"\n{q}")
    print(CH[CH["quantity"] == q].drop(columns="quantity").set_index("target").round(2).to_string())


SMR fleet, GW
         2050  RUN-1 2053  RUN-1 2055  _ext 2053  _ext 2055  RUN-1 2050->2055 %  _ext 2050->2055 %  RUN-1 - _ext 2055  2050 RUN-1 - _ext
target                                                                                                                                  
eia      20.7       22.07       22.99       20.7       20.7               11.06                0.0               2.29                0.0
aj       36.4       45.47       51.51       36.4       36.4               41.51                0.0              15.11                0.0
iaea     75.7       88.32       96.74       75.7       75.7               27.79                0.0              21.04                0.0
mck     103.4      124.03      137.79      103.4      103.4               33.26                0.0              34.39                0.0
cop28   203.4      249.53      280.29      203.4      203.4               37.80                0.0              76.89                0.0
eo      303.4      364.53 

C:\Users\ethan\AppData\Local\Temp\ipykernel_28936\711487343.py:6: PerformanceWarning: indexing past lexsort depth may impact performance.
  a, e = R.loc[(t, "RUN-1"), q], R.loc[(t, "_ext"), q]
